In [23]:
MODEL_NAME = "dinov2_vitb14"
IMG_SIZE = 224
print(MODEL_NAME)

dinov2_vitb14


In [24]:
import sys
from pathlib import Path

import torch
import torch.nn as nn

import matplotlib.pyplot as plt

from torchvision import transforms

In [25]:
PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print(PROJECT_ROOT)

d:\code\FSFM_Lite_Project


In [26]:
from src.datasets.celeba_spoof_dataset import CelebASpoofDataset, crop_face, read_bbox

print("Import OK")

Import OK


In [27]:
import pandas as pd

METADATA_ROOT = PROJECT_ROOT / "metadata"

train_df = pd.read_csv(METADATA_ROOT / "train_df.csv")

test_df = pd.read_csv(METADATA_ROOT / "test_df.csv")

print(train_df.shape)
print(test_df.shape)

(244274, 4)
(25758, 4)


In [28]:
from torchvision import transforms

face_transform = transforms.Compose(
    [
        transforms.ToPILImage(),
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
)

In [29]:
dataset = CelebASpoofDataset(train_df, transform=face_transform)

sample = dataset[0]

print(sample["image"].shape)

torch.Size([3, 224, 224])


In [30]:
dinov2 = torch.hub.load("facebookresearch/dinov2", "dinov2_vitb14")

dinov2.eval()

print("Loaded")

Using cache found in C:\Users\Admin/.cache\torch\hub\facebookresearch_dinov2_main


Loaded


In [31]:
image = sample["image"]

image = image.unsqueeze(0)

print(image.shape)

torch.Size([1, 3, 224, 224])


In [32]:
with torch.no_grad():

    features = dinov2.forward_features(image)

print(features.keys())

dict_keys(['x_norm_clstoken', 'x_norm_regtokens', 'x_norm_patchtokens', 'x_prenorm', 'masks'])


In [33]:
cls_token = features["x_norm_clstoken"]

print(cls_token.shape)

torch.Size([1, 768])


In [34]:
patch_tokens = features["x_norm_patchtokens"]

print(patch_tokens.shape)

torch.Size([1, 256, 768])


In [35]:
print("CLS:", cls_token.shape)

print("PATCH:", patch_tokens.shape)

CLS: torch.Size([1, 768])
PATCH: torch.Size([1, 256, 768])


In [36]:
with torch.no_grad():

    features = dinov2.forward_features(image)

cls_token = features["x_norm_clstoken"]

patch_tokens = features["x_norm_patchtokens"]

print(cls_token.shape)
print(patch_tokens.shape)

torch.Size([1, 768])
torch.Size([1, 256, 768])
